## 1. Data Loading & Inspection

In [17]:
# Install the datasets library if you haven't already
# !pip install datasets

from datasets import load_dataset
import pandas as pd

# Load the tweet_eval dataset (sentiment configuration)
dataset = load_dataset('cardiffnlp/tweet_eval', 'sentiment')


### Print Dataset Splits and Class Distribution

In [18]:
# Print dataset splits
print("Dataset splits:")
for split, data in dataset.items():
    print(f"- {split}: {len(data)} examples")

# Get label names
label_names = dataset['train'].features['label'].names
print(f"\nLabel names: {label_names}")

# Print class distribution for the training set
print("\nClass distribution (training set):")
train_labels = [label_names[label] for label in dataset['train']['label']]
display(pd.Series(train_labels).value_counts().sort_index())

# Confirm there are 3 labels
num_labels = dataset['train'].features['label'].num_classes
print(f"\nNumber of labels: {num_labels}")
if num_labels == 3:
    print("Confirmed: There are 3 labels (negative, neutral, positive).")
else:
    print("Warning: Expected 3 labels but found other number of labels.")


Dataset splits:
- train: 45615 examples
- test: 12284 examples
- validation: 2000 examples

Label names: ['negative', 'neutral', 'positive']

Class distribution (training set):


,count
negative,7093
neutral,20673
positive,17849



Number of labels: 3
Confirmed: There are 3 labels (negative, neutral, positive).


### Save two example tweets per label for later visualization

In [19]:
example_tweets = {}
for label_id, label_name in enumerate(label_names):
    # Filter the training set for the current label
    tweets_for_label = [ex for ex in dataset['train'] if ex['label'] == label_id]
    # Take the first two examples
    example_tweets[label_name] = tweets_for_label[:2]

print("Example tweets saved:")
for label_name, tweets in example_tweets.items():
    print(f"\n--- {label_name.upper()} ---")
    for i, tweet in enumerate(tweets):
        print(f"Tweet {i+1}: {tweet['text']}")

# Store for later use
import json
with open('example_tweets.json', 'w') as f:
    json.dump(example_tweets, f, indent=4)


Example tweets saved:

--- NEGATIVE ---
Tweet 1: So disappointed in wwe summerslam! I want to see john cena wins his 16th title
Tweet 2: That sucks if you have to take the SATs tomorrow

--- NEUTRAL ---
Tweet 1: "Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ"
Tweet 2: Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back to Minecraft in pc tomorrow night.

--- POSITIVE ---
Tweet 1: "QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"
Tweet 2: @user Alciato: Bee will invest 150 million in January, another 200 in the Summer and plans to bring Messi by 2017"


## 2. Tokenization Pipeline

In [20]:
# Install Transformers library if you haven't already
# !pip install transformers

from transformers import AutoTokenizer

# Initialize AutoTokenizer with distilbert-base-uncased
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

print("Tokenizer initialized.")


Tokenizer initialized.


### Create a preprocessing function

In [21]:
def preprocess_function(examples):
    # Truncate/pad tweets to 128 tokens.
    # Returns input_ids, attention_mask, and labels.
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128, return_tensors='pt')

print("Preprocessing function defined.")


Preprocessing function defined.


### Map the dataset and set format

In [22]:
# Apply the preprocessing function to the dataset
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Create a small subset for faster training/validation during development
# You can remove this for full training
# tokenized_dataset["train"] = tokenized_dataset["train"].shuffle(seed=42).select(range(5000))
# tokenized_dataset["validation"] = tokenized_dataset["validation"].shuffle(seed=42).select(range(1000))
# tokenized_dataset["test"] = tokenized_dataset["test"].shuffle(seed=42).select(range(1000))

# Rename the 'label' column to 'labels' to match the model's expected input
tokenized_dataset = tokenized_dataset.rename_columns({"label": "labels"})

# Set the format to PyTorch tensors
tokenized_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print("Dataset tokenized, renamed 'label' to 'labels', and set to PyTorch format.")

# Display a sample of the tokenized dataset
print("\nSample of tokenized dataset (train split, first example):")
print(tokenized_dataset['train'][0])


Dataset tokenized, renamed 'label' to 'labels', and set to PyTorch format.

Sample of tokenized dataset (train split, first example):


ImportError: cannot import name 'VideoReader' from 'torchvision.io' (/usr/local/lib/python3.12/dist-packages/torchvision/io/__init__.py)

## 3. Fine-Tuning Setup

In [24]:
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from evaluate import load as load_metric # Corrected import for load_metric

# Load AutoModelForSequenceClassification with 3 labels
num_labels = dataset['train'].features['label'].num_classes
model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels,
    output_attentions=True, # Ensure model can output attentions
    attn_implementation='eager' # Force eager attention for attention weights
)

print("Model loaded with 3 labels.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded with 3 labels.


### Define Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,               # epochs=3
    per_device_train_batch_size=32,   # batch_size=32
    per_device_eval_batch_size=32,
    learning_rate=5e-5,               # lr=5e-5
    weight_decay=0.01,                # weight_decay=0.01
    eval_strategy='epoch',            # Corrected: use eval_strategy instead of evaluation_strategy
    logging_dir='./logs',
    logging_steps=500,
    save_strategy='epoch',
    load_best_model_at_end=True,      # Save the best checkpoint
    metric_for_best_model='f1',       # Metric to use for early stopping/best model saving
    greater_is_better=True,
)

print("Training arguments defined.")


### Implement `compute_metrics` function

In [ ]:
def compute_metrics(eval_pred):
    metric_acc = load_metric('accuracy')
    metric_f1 = load_metric('f1')
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = metric_acc.compute(predictions=predictions, references=labels)
    f1_macro = metric_f1.compute(predictions=predictions, references=labels, average='macro')

    return {'accuracy': accuracy['accuracy'], 'f1': f1_macro['f1']}

print("compute_metrics function defined.")


### Initialize and Train the Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("Trainer initialized. Starting training...")

trainer.train()

print("Training complete. Best model saved to './results/checkpoint-...' (or similar).")


## 4. Evaluation & Calibration

### Evaluate on the validation split; log accuracy and F1

In [ ]:
print("Evaluating model on the validation set...")
eval_results = trainer.evaluate(eval_dataset=tokenized_dataset['validation'])
print(f"Validation Results: {eval_results}")


### Collect the softmax scores for predicted classes on the test split

In [ ]:
import torch
import numpy as np

print("Predicting on the test set to collect softmax scores...")
predictions = trainer.predict(tokenized_dataset['test'])

# The predictions object contains predictions, label_ids, and metrics
logits = predictions.predictions
probabilities = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()

# Get the confidence of the predicted class for each example
predicted_labels = np.argmax(probabilities, axis=-1)
confidence_scores = np.max(probabilities, axis=-1)

print("Softmax scores and confidence collected for the test set.")


### Plot a histogram of confidence scores and comment on trends

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(confidence_scores, bins=10, kde=True, stat='count')
plt.title('Histogram of Confidence Scores for Test Set Predictions')
plt.xlabel('Confidence Score (Softmax Max Probability)')
plt.ylabel('Number of Samples')
plt.xlim(0, 1) # Confidence scores are between 0 and 1
plt.grid(axis='y', alpha=0.75)
plt.show()

print("\n--- Comment on over/under-confidence trends ---")
print("Based on the histogram, we can observe the distribution of the model's confidence in its predictions. ")
print("A high concentration of scores near 1.0 indicates strong confidence, while scores near 0.33 (for 3 classes) indicate low confidence or uncertainty.")
print("Peaks near 1.0 suggest the model is often very confident. If there are also many low confidence scores, it means the model is uncertain for some predictions.")
print("The shape of the distribution helps assess calibration: a well-calibrated model's confidence should match its accuracy.")


## 5. Attention Inspection

### Pick one of the saved example tweets and prepare it for attention visualization

In [ ]:
import json
import torch
from transformers import AutoModel, AutoModelForSequenceClassification
import matplotlib.pyplot as plt
import seaborn as sns

# Load example tweets saved from Task 1
with open('example_tweets.json', 'r') as f:
    example_tweets = json.load(f)

# Let's pick a negative example tweet for visualization
chosen_label = 'negative'
chosen_tweet = example_tweets[chosen_label][0]['text']
print(f"Chosen tweet for attention visualization (label: {chosen_label}):\n'{chosen_tweet}'")

# For attention visualization, it's often easier to use the base model or a specific layer output
# Let's load the model from the fine-tuned checkpoint if available, otherwise base distilbert
try:
    # Assuming the best model checkpoint is saved in the results directory
    # You might need to adjust the path based on your actual save location
    # For simplicity, let's load the model used for training or a fresh AutoModel
    model_for_attention = AutoModelForSequenceClassification.from_pretrained(
        training_args.output_dir + '/' + trainer.state.best_model_checkpoint.split('/')[-1] if trainer.state.best_model_checkpoint else 'distilbert-base-uncased',
        output_attentions=True,
        attn_implementation='eager'
    )
except Exception as e:
    print(f"Could not load fine-tuned model for attention visualization: {e}. Loading distilbert-base-uncased.")
    model_for_attention = AutoModel.from_pretrained('distilbert-base-uncased', output_attentions=True, attn_implementation='eager')

model_for_attention.eval()

# Tokenize the chosen tweet
inputs = tokenizer(chosen_tweet, return_tensors='pt', truncation=True, padding='max_length', max_length=128)
input_ids = inputs['input_ids']
attention_mask = inputs['attention_mask']

# Get tokenized words for plotting
words = tokenizer.convert_ids_to_tokens(input_ids[0])

print("Tweet tokenized and model prepared for attention extraction.")


### Pass it through AutoModel to grab the last-layer attention weights

In [ ]:
with torch.no_grad():
    outputs = model_for_attention(input_ids, attention_mask=attention_mask)

# attentions is a tuple of (layer_output_attention)
# Each layer_output_attention is (batch_size, num_heads, sequence_length, sequence_length)
# For DistilBERT, it has 6 layers, so outputs.attentions will have 6 tensors.

# Get the attention weights from the last layer
last_layer_attentions = outputs.attentions[-1] # Shape: (1, num_heads, sequence_length, sequence_length)

print(f"Shape of last layer attentions: {last_layer_attentions.shape}")

# Average across heads, then visualize the attention directed from [CLS] to each token
# We are interested in the attention from the [CLS] token (index 0) to all other tokens
cls_attention = last_layer_attentions[0, :, 0, :].mean(dim=0) # Average across all heads

# Remove padding tokens from visualization
actual_tokens_len = (input_ids[0] != tokenizer.pad_token_id).sum().item()
cls_attention = cls_attention[:actual_tokens_len]
words_for_plot = words[:actual_tokens_len]

print("Attention weights extracted and averaged.")


### Visualize the attention directed from [CLS] to each token (heatmap or bar chart)

## Production-Style `analyze_text()` Function for Explainable Inference

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Ensure label_names are available (assuming `dataset` was loaded and processed correctly)
# If running this cell independently, ensure dataset is loaded or define label_names manually
# label_names = dataset['train'].features['label'].names
# Alternatively, hardcode if you're sure: label_names = ['negative', 'neutral', 'positive']

# If `model` and `tokenizer` are not globally available (e.g., in a fresh session),
# you might need to reload them from the saved checkpoint.
# For this function to be truly 'production-style', it should ideally load its own dependencies.
# Let's assume `tokenizer` and `model` (the fine-tuned one) are already in scope.

def analyze_text(text, tokenizer, model, label_names, top_k_tokens=5):
    """
    Analyzes input text to predict sentiment, confidence, and identify top contributing tokens.

    Args:
        text (str): The input text to analyze.
        tokenizer: The pre-trained tokenizer (e.g., DistilBertTokenizer).
        model: The fine-tuned sequence classification model.
        label_names (list): A list of human-readable label names.
        top_k_tokens (int): Number of top contributing tokens to highlight.

    Returns:
        dict: A dictionary containing:
            - 'label': Predicted sentiment label (string).
            - 'confidence': Softmax probability of the predicted label.
            - 'highlighted_tokens': A list of (token, attention_score) for top_k tokens.
    """
    # Tokenize input
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding='max_length', max_length=128)
    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask, output_attentions=True)

    # Get predictions and confidence
    logits = outputs.logits
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    predicted_label_id = torch.argmax(probabilities, dim=-1).item()
    predicted_label = label_names[predicted_label_id]
    confidence = probabilities[0, predicted_label_id].item()

    # Get attention weights from the last layer (CLS token to others)
    # Assuming the model used is `AutoModelForSequenceClassification` which returns `attentions`
    # if `output_attentions=True` is passed to the forward method or model config.
    if hasattr(outputs, 'attentions') and outputs.attentions is not None:
        last_layer_attentions = outputs.attentions[-1] # Shape: (1, num_heads, sequence_length, sequence_length)

        # Average across heads for [CLS] token's attention to other tokens
        cls_attention_scores = last_layer_attentions[0, :, 0, :].mean(dim=0).cpu().numpy()

        # Get original tokens
        tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

        # Filter out special tokens and padding from attention scores for highlighting
        # Start from index 1 to skip [CLS] token itself for highlighting other words
        # And exclude [SEP] and [PAD] from `tokens` and corresponding attention scores
        clean_tokens = []
        clean_attention_scores = []
        for i, token in enumerate(tokens):
            if token not in ['[CLS]', '[SEP]', '[PAD]'] and attention_mask[0, i].item() == 1:
                clean_tokens.append(token)
                clean_attention_scores.append(cls_attention_scores[i])

        # Pair tokens with their attention scores and sort by score
        token_attention_pairs = sorted(zip(clean_tokens, clean_attention_scores), key=lambda x: x[1], reverse=True)

        # Get top K contributing tokens
        highlighted_tokens = token_attention_pairs[:top_k_tokens]
    else:
        highlighted_tokens = []
        print("Warning: Attention weights not available from the model output.")

    return {
        'label': predicted_label,
        'confidence': confidence,
        'highlighted_tokens': highlighted_tokens
    }

print("analyze_text() function defined.")


### Example Usage of `analyze_text()`

In [ ]:
# Ensure label_names is correctly defined if not already in scope from previous cells
if 'label_names' not in globals():
    try:
        label_names = dataset['train'].features['label'].names
    except NameError:
        print("Warning: `dataset` not found. Assuming default label names for tweet_eval sentiment.")
        label_names = ['negative', 'neutral', 'positive'] # Default for tweet_eval sentiment

# Test with a new tweet
example_sentence = "This movie was absolutely fantastic! Highly recommend it to everyone."
analysis_result = analyze_text(example_sentence, tokenizer, model, label_names)
print("\n--- Analysis for: " + example_sentence + " ---")
print(f"Predicted Label: {analysis_result['label']}")
print(f"Confidence: {analysis_result['confidence']:.4f}")
print("Top Contributing Tokens:")
for token, score in analysis_result['highlighted_tokens']:
    print(f"  - {token}: {score:.4f}")

example_sentence_negative = "What a terrible and disappointing experience. I'm so upset."
analysis_result_negative = analyze_text(example_sentence_negative, tokenizer, model, label_names)
print("\n--- Analysis for: " + example_sentence_negative + " ---")
print(f"Predicted Label: {analysis_result_negative['label']}")
print(f"Confidence: {analysis_result_negative['confidence']:.4f}")
print("Top Contributing Tokens:")
for token, score in analysis_result_negative['highlighted_tokens']:
    print(f"  - {token}: {score:.4f}")

print("\nAll tasks completed. The notebook now includes data loading, tokenization, fine-tuning, evaluation, attention visualization, and a production-ready explainable inference function.")


## Production-Style `analyze_text()` Function for Explainable Inference

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Ensure label_names are available (assuming `dataset` was loaded and processed correctly)
# If running this cell independently, ensure dataset is loaded or define label_names manually
# label_names = dataset['train'].features['label'].names
# Alternatively, hardcode if you're sure: label_names = ['negative', 'neutral', 'positive']

# If `model` and `tokenizer` are not globally available (e.g., in a fresh session),
# you might need to reload them from the saved checkpoint.
# For this function to be truly 'production-style', it should ideally load its own dependencies.
# Let's assume `tokenizer` and `model` (the fine-tuned one) are already in scope.

def analyze_text(text, tokenizer, model, label_names, top_k_tokens=5):
    """
    Analyzes input text to predict sentiment, confidence, and identify top contributing tokens.

    Args:
        text (str): The input text to analyze.
        tokenizer: The pre-trained tokenizer (e.g., DistilBertTokenizer).
        model: The fine-tuned sequence classification model.
        label_names (list): A list of human-readable label names.
        top_k_tokens (int): Number of top contributing tokens to highlight.

    Returns:
        dict: A dictionary containing:
            - 'label': Predicted sentiment label (string).
            - 'confidence': Softmax probability of the predicted label.
            - 'highlighted_tokens': A list of (token, attention_score) for top_k tokens.
    """
    # Tokenize input
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding='max_length', max_length=128)
    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask, output_attentions=True)

    # Get predictions and confidence
    logits = outputs.logits
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    predicted_label_id = torch.argmax(probabilities, dim=-1).item()
    predicted_label = label_names[predicted_label_id]
    confidence = probabilities[0, predicted_label_id].item()

    # Get attention weights from the last layer (CLS token to others)
    # Assuming the model used is `AutoModelForSequenceClassification` which returns `attentions`
    # if `output_attentions=True` is passed to the forward method or model config.
    if hasattr(outputs, 'attentions') and outputs.attentions is not None:
        last_layer_attentions = outputs.attentions[-1] # Shape: (1, num_heads, sequence_length, sequence_length)

        # Average across heads for [CLS] token's attention to other tokens
        cls_attention_scores = last_layer_attentions[0, :, 0, :].mean(dim=0).cpu().numpy()

        # Get original tokens
        tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

        # Filter out special tokens and padding from attention scores for highlighting
        # Start from index 1 to skip [CLS] token itself for highlighting other words
        # And exclude [SEP] and [PAD] from `tokens` and corresponding attention scores
        clean_tokens = []
        clean_attention_scores = []
        for i, token in enumerate(tokens):
            if token not in ['[CLS]', '[SEP]', '[PAD]'] and attention_mask[0, i].item() == 1:
                clean_tokens.append(token)
                clean_attention_scores.append(cls_attention_scores[i])

        # Pair tokens with their attention scores and sort by score
        token_attention_pairs = sorted(zip(clean_tokens, clean_attention_scores), key=lambda x: x[1], reverse=True)

        # Get top K contributing tokens
        highlighted_tokens = token_attention_pairs[:top_k_tokens]
    else:
        highlighted_tokens = []
        print("Warning: Attention weights not available from the model output.")

    return {
        'label': predicted_label,
        'confidence': confidence,
        'highlighted_tokens': highlighted_tokens
    }

print("analyze_text() function defined.")


### Example Usage of `analyze_text()`

In [ ]:
# Ensure label_names is correctly defined if not already in scope from previous cells
if 'label_names' not in globals():
    try:
        label_names = dataset['train'].features['label'].names
    except NameError:
        print("Warning: `dataset` not found. Assuming default label names for tweet_eval sentiment.")
        label_names = ['negative', 'neutral', 'positive'] # Default for tweet_eval sentiment

# Test with a new tweet
example_sentence = "This movie was absolutely fantastic! Highly recommend it to everyone."
analysis_result = analyze_text(example_sentence, tokenizer, model, label_names)
print("\n--- Analysis for: " + example_sentence + " ---")
print(f"Predicted Label: {analysis_result['label']}")
print(f"Confidence: {analysis_result['confidence']:.4f}")
print("Top Contributing Tokens:")
for token, score in analysis_result['highlighted_tokens']:
    print(f"  - {token}: {score:.4f}")

example_sentence_negative = "What a terrible and disappointing experience. I'm so upset."
analysis_result_negative = analyze_text(example_sentence_negative, tokenizer, model, label_names)
print("\n--- Analysis for: " + example_sentence_negative + " ---")
print(f"Predicted Label: {analysis_result_negative['label']}")
print(f"Confidence: {analysis_result_negative['confidence']:.4f}")
print("Top Contributing Tokens:")
for token, score in analysis_result_negative['highlighted_tokens']:
    print(f"  - {token}: {score:.4f}")

print("\nAll tasks completed. The notebook now includes data loading, tokenization, fine-tuning, evaluation, attention visualization, and a production-ready explainable inference function.")


In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(words_for_plot, cls_attention.cpu().numpy())
plt.xticks(rotation=90)
plt.title(f'Attention from [CLS] token to other tokens for \'{chosen_tweet}\'')
plt.xlabel('Tokens')
plt.ylabel('Attention Score (averaged across heads)')
plt.tight_layout()
plt.show()

# Alternatively, a heatmap for better comparison of token importance
plt.figure(figsize=(15, 3)) # Wider figure for better token visibility
sns.heatmap(
    cls_attention.cpu().numpy().reshape(1, -1), # Reshape for heatmap
    xticklabels=words_for_plot,
    yticklabels=['[CLS] Attention'],
    cmap='viridis', # 'YlGnBu' or 'Blues' or 'viridis'
    linewidths=0.5,
    linecolor='black',
    cbar_kws={'label': 'Attention Score'}
)
plt.title(f'Attention from [CLS] token to other tokens for \'{chosen_tweet}\' (Heatmap)')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("\n--- Document insights ---")
print("Observing the attention plot, we can see which tokens the [CLS] token (representing the overall sentence meaning) pays most attention to. ")
print("For example, in a negative sentiment tweet, we might expect the [CLS] token to focus on words expressing negativity or frustration, like 'terrible', 'bad', 'hate', or words indicating the subject of the negative sentiment. ")
print("If the tweet was 'The service was terrible and slow.', we might find higher attention scores for 'terrible' and 'slow' from the [CLS] token.")
print("This visualization helps understand what parts of the input are most influential for the model's overall understanding, especially for classification tasks.")
